# Building a Real Deep Learning Image Classifier

In this lecture, we will build a complete deep learning system for handwritten letter recognition using a modern convolutional neural network.

The goal is to classify images of letters (A–Z) from the EMNIST dataset. Each image contains a single handwritten character, and the model must predict which letter it represents.

This is a classic **image classification problem**, and it is a great example for learning how deep learning systems are built in practice.

---

## What We Will Learn

Instead of focusing only on theory, we will go through the full pipeline used in real-world deep learning projects:

* How to structure and load an image dataset
* How to preprocess images for a pretrained neural network
* How to use a pretrained model (ResNet18) for a new task
* How to train only part of a model (transfer learning)
* How to fine-tune a pretrained network for better performance
* How to evaluate models using meaningful metrics
* How to interpret results using confusion matrices

---

## Why This Problem Is Important

Handwritten character recognition is a historically important problem in computer vision. It appears in:

* postal mail sorting
* digitizing handwritten documents
* OCR (optical character recognition) systems
* real-world document automation pipelines

Even though modern systems are highly accurate, the core ideas behind them are the same as what we will build here.

---

## Why We Use Deep Learning

Traditional machine learning methods struggle with image data because they require manual feature engineering.

Deep learning solves this by learning features automatically:

* early layers detect edges and textures
* deeper layers detect shapes and patterns
* final layers make classification decisions

This hierarchical learning makes convolutional neural networks especially powerful for image tasks.

---

## Why We Use Transfer Learning

Training deep neural networks from scratch requires large datasets and significant computational power.

Instead, we use **transfer learning**, which means:

> We start from a model already trained on a large dataset (ImageNet) and adapt it to our task.

This allows us to:

* train faster
* achieve better performance with less data
* avoid overfitting on small datasets

---

## What We Will Build

By the end of this lecture, we will have a complete system that:

* takes a handwritten letter image as input
* processes it through a pretrained ResNet18 model
* outputs one of 26 classes (A–Z)
* evaluates performance using multiple metrics

This pipeline is very similar to what is used in real-world computer vision applications.


In [ ]:
import copy
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
#import torchinfo
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)


## 1. Reproducibility and Device Setup

To ensure consistent results across runs, we fix all random seeds:

* Python random
* NumPy
* PyTorch (CPU and GPU)

We also define a unified device selection strategy:

```python
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
```

### Why this matters

Deep learning training is stochastic. Without fixed seeds:

* results may vary across runs
* debugging becomes difficult
* experiments are not comparable


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
seed_everything()

# Standardized to lowercase 'device' globally across all loops
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"device is {device}")


## 2. Dataset Design

We implement a custom dataset class that reads images from folder structure:

```
root/
  A/
  B/
  C/
  ...
```

Each folder represents a class label.

### Key idea

The dataset class must implement:

* `__len__()` → total number of samples
* `__getitem__()` → returns (image, label)


In [ ]:
class EMNISTFolderDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and not d.startswith('.')
        ])

        self.class_to_idx = {
            cls_name: i
            for i, cls_name in enumerate(self.classes)
        }

        self.samples = []

        for cls_name in self.classes:

            cls_dir = os.path.join(root_dir, cls_name)

            for img_name in os.listdir(cls_dir):

                if img_name.endswith(".png"):

                    img_path = os.path.join(cls_dir, img_name)

                    self.samples.append(
                        (img_path, self.class_to_idx[cls_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image, label


## 3. Data Preprocessing Pipeline

Since ResNet18 was trained on ImageNet, we must match its expected input format:

### Step 1: Resize

```python
Resize((224, 224))
```

### Step 2: Convert grayscale → RGB-like format

```python
Grayscale(num_output_channels=3)
```

### Step 3: Normalize using ImageNet statistics

```python
Normalize(mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225])
```

### Important concept

> Pretrained models expect the same data distribution they were trained on.


In [ ]:
# ResNet18 was pretrained on ImageNet images of size 224x224
# Duplicate grayscale channel 3 times
# so pretrained ResNet can accept the images
emnist_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

#emnist_trainval_dataset = EMNISTFolderDataset("data/emnist_letters_subset/train", transform=emnist_transforms)
#emnist_test_dataset     = EMNISTFolderDataset("data/emnist_letters_subset/test",  transform=emnist_transforms)
emnist_trainval_dataset = EMNISTFolderDataset(
    "/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets/EMNIST_letters_subset/train",
    #"data/emnist_letters_subset/train",
    transform=emnist_transforms
)

emnist_test_dataset = EMNISTFolderDataset(
    "/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets/EMNIST_letters_subset/test",
    #"data/emnist_letters_subset/test",
    transform=emnist_transforms
)


## 4. Train / Validation / Test Split

We split the dataset into:

* 80% training
* 20% validation
* separate test set

### Why validation set?

It allows us to:

* detect overfitting
* tune hyperparameters
* decide when to stop training


In [ ]:
# 80/20 train/val split logic
total_size = len(emnist_trainval_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

emnist_train_dataset, emnist_val_dataset = random_split(
    emnist_trainval_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

print(f"Train samples: {len(emnist_train_dataset)}")
print(f"Validation samples: {len(emnist_val_dataset)}")
print(f"Test samples: {len(emnist_test_dataset)}")


In [ ]:
# DATALOADERS

batch_size = 128
loader_args = {"batch_size": batch_size, "num_workers": 0}

emnist_train_loader = DataLoader(emnist_train_dataset, shuffle=True, **loader_args)
emnist_val_loader   = DataLoader(emnist_val_dataset, shuffle=False, **loader_args)
emnist_test_loader  = DataLoader(emnist_test_dataset, shuffle=False, **loader_args)


In [ ]:
def visualize_dataset(dataset, num_samples=8):
    # ImageNet normalization constants
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 3))
    total_samples = len(dataset)
    random_indices = random.sample(range(total_samples), num_samples)
    # --- SAFETY GUARD FOR SUBSETS ---
    # If it's a random_split Subset, unpack the original class list
    if hasattr(dataset, "dataset"):
        classes = dataset.dataset.classes
    else:
        classes = dataset.classes
    # --------------------------------

    for i, idx in enumerate(random_indices):
        image_tensor, label_idx = dataset[idx]
        img = image_tensor.numpy().transpose((1, 2, 0))
        img = std * img + mean
        img = np.clip(img, 0, 1)
        # Uses the safely resolved classes array
        class_letter = classes[label_idx]
        axes[i].imshow(img)
        axes[i].set_title(f"Label: {class_letter}", fontsize=11, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
    

In [ ]:
# Call it directly on your subset dataset
visualize_dataset(emnist_train_dataset, num_samples=8)


## 5. Model: ResNet18 Transfer Learning

We load a pretrained ResNet18 and replace the final layer:

```python
model.fc = nn.Linear(model.fc.in_features, 26)
```

### Freezing strategy

Initially:

* freeze all backbone layers
* train only classifier head

Later:

* unfreeze last ResNet block (`layer4`)
* fine-tune with smaller learning rate

### Why this works

Early layers learn:

* edges
* textures

Later layers learn:

* task-specific representations


In [ ]:
# 5. LOAD PRETRAINED RESNET18

# 1. Initialize the model architecture with no pre-trained weights
model = resnet18(weights=None)

# 2. Define the exact path where you saved the .pth file
weights_path = "/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets/ResNet18_Weights/resnet18-f37072fd.pth"

# 3. Load the state dictionary manually
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)

print("Model weights loaded successfully from local storage!")

# Replace final classification layer
model.fc = nn.Linear(model.fc.in_features, 26)

# Freeze pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Train only classifier first
for param in model.fc.parameters():
    param.requires_grad = True

#print(torchinfo.summary(model, input_size=(batch_size, 3, 224, 224)))


## 6. Early Stopping

We stop training when validation performance stops improving.

### Why?

To prevent:

* overfitting
* unnecessary computation

We restore the best model based on validation loss.


In [ ]:
# 6. TRAINING UTILITIES

class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_acc = -float("inf")
        self.best_weights = None
        self.best_epoch = 0

    def step(self, model, val_loss, val_acc, epoch):

        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.best_epoch = epoch+1
            self.counter = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        self.counter += 1
        print(
            f"[EarlyStopping] Epoch {epoch+1 if epoch is not None else ''}: "
            f"No improvement → counter {self.counter}/{self.patience}"
        )
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)


In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == y).sum().item()

            total += y.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


## 7. Training Strategy

We use:

* CrossEntropyLoss (multi-class classification)
* Adam optimizer
* mini-batch gradient descent

### Key idea

Each iteration:

1. Forward pass
2. Compute loss
3. Backpropagation
4. Update weights


In [ ]:
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

early_stopping = EarlyStopping(patience=3)


## 8. Fine-Tuning Strategy

At epoch 3:

* Unfreeze `layer4`
* Use **differential learning rates**:

```python
fc:       lr = 1e-4
layer4:   lr = 1e-5
```

### Why different learning rates?

* classifier is random → needs larger updates
* pretrained features already good → should change slowly


In [ ]:
# ------------------------------------------------------------
# Store metrics for plotting
# ------------------------------------------------------------
epochs = 30

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(epochs):

    # --------------------------------------------------------
    # Fine-tuning phase:
    # unfreeze only the last ResNet block + classifier
    # --------------------------------------------------------

    if epoch == 3:
        print("\nUnfreezing for fine-tuning...\n")

        # Unfreeze only the last convolutional block
        for param in model.layer4.parameters():
            param.requires_grad = True

        # Classifier was already trainable,
        # but this makes the intent explicit
        for param in model.fc.parameters():
            param.requires_grad = True

        # Different learning rates:
        # - slightly larger LR for new classifier
        # - smaller LR for pretrained features
        # Re-initializing torch.optim.Adam completely wipes out 
        # the running averages of past gradients (momentum states) 
        # built up during the first 3 epochs. 
        # The optimizer essentially gets "amnesia" and has to 
        # relearn the gradient velocities for the classifier from scratch, 
        # alongside the newly unfrozen backbone.

        optimizer = torch.optim.Adam([
            {"params": model.fc.parameters(), "lr": 1e-4},
            {"params": model.layer4.parameters(), "lr": 1e-5},
        ])

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    total_train_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(emnist_train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        # More correct weighted averaging
        total_train_loss += loss.item() * x.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()

        total += y.size(0)

    # --------------------------------------------------------
    # Compute training metrics
    # --------------------------------------------------------

    train_loss = total_train_loss / total
    train_accuracy = correct / total

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_loss, val_accuracy = evaluate(model, emnist_val_loader, criterion, device)

    # --------------------------------------------------------
    # Store metrics
    # --------------------------------------------------------

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # --------------------------------------------------------
    # Epoch summary
    # --------------------------------------------------------

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if early_stopping.step(model, val_loss, val_accuracy, epoch):
        print("\nEarly stopping triggered.")
        break


# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------

early_stopping.restore(model)

print("\n================================================")
print("BEST MODEL SUMMARY")
print("================================================")
print(f"Best Epoch         : {early_stopping.best_epoch}")
print(f"Best Validation Acc: {early_stopping.best_acc:.4f}")


## 9. VISUALIZE TRAINING CURVES

We plot:
- training loss
- validation loss
- training accuracy
- validation accuracy

These plots help diagnose:
- overfitting
- underfitting
- convergence behavior


In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
best_epoch = early_stopping.best_epoch
best_loss = early_stopping.best_loss
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Cross Entropy Loss", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
epochs_range = range(1, len(train_losses) + 1)
best_epoch = early_stopping.best_epoch
best_acc = early_stopping.best_acc

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_accuracies, label="Training Accuracy", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
plt.scatter(best_epoch, best_acc, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Accuracy", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Accuracy", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


## 10. Evaluation Metrics

We evaluate using:

### Accuracy

Simple correctness measure.

### F1 Score (Macro & Weighted)

* Macro F1: treats all classes equally
* Weighted F1: accounts for class imbalance

> Accuracy alone can hide poor performance on minority classes.


In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for x, y in emnist_test_loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        predictions = logits.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            y.cpu().numpy()
        )

accuracy = np.mean(
    np.array(all_predictions) == np.array(all_labels)
)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n================================================")
print("FINAL TEST METRICS")
print("================================================")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1   : {f1_weighted:.4f}")


## 11. Confusion Matrix

A confusion matrix shows which classes are commonly confused.

This is especially important for handwritten letters:

Common confusions:

* O vs Q
* I vs L
* C vs G


In [ ]:
class_names = [
    chr(i)
    for i in range(ord("A"), ord("Z") + 1)
]

cm = confusion_matrix(
    all_labels,
    all_predictions
)

fig, ax = plt.subplots(figsize=(10, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    xticks_rotation=90,
    ax=ax,
    colorbar=False
)

ax.grid(False)
ax.set_xticklabels(class_names, rotation=0)

plt.title("EMNIST Letters Confusion Matrix")
plt.show()


## 12. Classification Report

We compute per-class metrics:

* precision
* recall
* F1-score

### Why this matters

It helps identify:

* which letters are learned well
* which letters are systematically misclassified


In [ ]:
print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

## 13. Key Takeaways

* Transfer learning allows fast training with limited data
* Pretrained CNNs generalize well to new visual tasks
* Freezing and fine-tuning is a two-stage optimization strategy
* Evaluation must go beyond accuracy
* Confusion matrices reveal model weaknesses that metrics hide



## Exercise: MNIST $\rightarrow$ EMNIST Transfer Learning with a Custom CNN

In this exercise, you will build a complete deep learning workflow that demonstrates how knowledge learned from one task can be reused for another.

You will first train a convolutional neural network (CNN) on handwritten digits (MNIST), and then transfer the learned visual features to a new task: handwritten letter classification (EMNIST).

This process is known as **transfer learning**, and it is one of the most important techniques in modern deep learning.



## Part 1 — Train a CNN on MNIST (Source Task)

First train a simple CNN on the MNIST dataset, which contains handwritten digits (0–9).

* Train the model to correctly classify digits

Hint:

```python
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.features = nn.Sequential(...)

        self.classifier = nn.Sequential(...)

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

```

## Part 2 — Save Learned Features

After training on MNIST, save the learned weights.

* Save model parameters correctly
* Separate feature extractor weights from classifier weights

This step simulates a real-world workflow where models are:

* trained once on a large dataset
* reused later for different tasks

Hint:

`torch.save(model.features.state_dict(), "mnist_features.pth")`

## Part 3 — Load EMNIST (Target Task)

Now switch to a new dataset: EMNIST letters (A–Z).

* Load EMNIST dataset
* Prepare data loaders for training, validation, and testing

This step introduces a **domain shift**:

> The model must adapt from digits → letters


## Part 4 — Transfer Learned Features

Initialize a new model and load pretrained convolutional weights from MNIST.

* Reuse convolutional layers from MNIST training
* Replace the final classification layer (10 → 26 classes)

Hint:

`model.features.load_state_dict(torch.load("mnist_features.pth"))`

## Part 5 — Freeze Feature Extractor

Initially, freeze the convolutional backbone.

* pretrained features are kept fixed
* only the classifier is trained

You need to:

* Train only the classification head
* Observe how well fixed features perform on EMNIST


## Part 6 — Fine-Tuning the Model

After several epochs, unfreeze the convolutional layers and continue training.

This allows the model to:

* adjust pretrained features to the new dataset
* improve performance beyond feature extraction alone

You need to:

* Unfreeze convolutional backbone
* Use different learning rates for:

  * classifier (higher learning rate)
  * feature extractor (lower learning rate)


## Part 7 — Early Stopping

Use early stopping to prevent overfitting.

* Monitor validation loss
* Stop training when performance stops improving
* Restore the best model

> The best model is not always the last one trained.


## Part 8 — Evaluate Model Performance

After training, evaluate the model on the test set.

Compute:

* Accuracy → overall correctness
* Macro F1 score → equal importance for all classes
* Weighted F1 score → accounts for class imbalance


## Part 9 — Analyze Confusion Matrix

Visualize prediction errors using a confusion matrix.

This helps identify which letters are most frequently confused.

* Identify systematic errors
* Understand which classes are difficult for the model


## Part 10 — Classification Report

Generate a per-class performance report including:

* precision
* recall
* F1-score

You need to:

* Identify strong and weak classes
* Analyze whether performance is uniform across all letters
